# 03 · Market matching — PM ↔ sportsbook, auditable

**Purpose.** Link the two market universes on teams + kickoff +
competition + market type + line + period + **settlement basis** —
never fuzzy names alone. Output: accepted / rejected / ambiguous tables,
a manual-override file, and the gold `market_links` dataset.

**Hard rule encoded here (and unit-tested):** sportsbook 1X2 settles at
90 minutes; PM advancement includes ET+pens. Different settlement ⇒
DIFFERENT market, always rejected.

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
JB = next(q for q in [_here, *_here.parents] if (q / "lib" / "bootstrap.py").exists())
sys.path.insert(0, str(JB))

import datetime as dt
import pandas as pd
import polars as pl
import lib.bootstrap as bt
import lib.config as cfg
import lib.storage as st
import lib.matching as mtch
import lib.ids as ids

manifest = bt.run_manifest("03_market_matching")
p = cfg.load_params()

## 1 · Side A — Polymarket canonical match markets (from notebook 02)

In [2]:
mk = st.load_dataset("bronze", "pm_markets")
pm_side = mtch.pm_canonical_matches(mk)
print(f"PM canonical 1X2 markets: {pm_side.height}")
pm_side.tail(5).to_pandas()

PM canonical 1X2 markets: 91


,source,source_market_id,home,away,kickoff_utc,market_type,line,period,settlement
0,polymarket,fifwc-usa-aus-2026-06-19,United States,Australia,2026-06-19 19:00:00+00:00,1x2,NaN,FT,90min
1,polymarket,fifwc-usa-bel-2026-07-06,United States,Belgium,2026-07-07 00:00:00+00:00,1x2,NaN,FT,90min
2,polymarket,fifwc-usa-bih-2026-07-01,United States,Bosnia and Herzegovina,2026-07-02 00:00:00+00:00,1x2,NaN,FT,90min
3,polymarket,fifwc-usa-par-2026-06-12,United States,Paraguay,2026-06-13 01:00:00+00:00,1x2,NaN,FT,90min
4,polymarket,fifwc-uzb-col-2026-06-17,Uzbekistan,Colombia,2026-06-18 02:00:00+00:00,1x2,NaN,FT,90min


## 2 · Side B — sportsbook markets
Preferred: silver `sportsbook_quotes` from notebook 01 (live Odds API).
Fallback when 01 couldn't pull: the repo schedule feed
(`site/scores_markets.json`) as a REFERENCE side — clearly labelled; it
proves the matcher end-to-end but carries no odds.

In [3]:
sb_rows, sb_source = [], None
try:
    sq = st.load_dataset("silver", "sportsbook_quotes")
    sb_source = "theoddsapi silver (notebook 01)"
    for r in (sq.group_by("event_id", "market_id")
                .agg(pl.col("kickoff_utc").first(),
                     pl.col("outcome").first(),
                     pl.col("line").first(),
                     pl.col("source_event_id").first())).to_dicts():
        parsed = ids.parse_event_id(r["event_id"])
        sb_rows.append({
            "source": "theoddsapi", "source_market_id": r["market_id"],
            "home": parsed["home_slug"].replace("-", " "),
            "away": parsed["away_slug"].replace("-", " "),
            "kickoff_utc": r["kickoff_utc"],
            "market_type": r["market_id"].split("|")[1],
            "line": r["line"], "period": "FT", "settlement": ids.S_90MIN})
except FileNotFoundError:
    sb_source = "REFERENCE side: site/scores_markets.json schedule (no odds — matcher demo)"
    import json
    sm = json.loads(pathlib.Path(bt.SCORES_MARKETS_JSON).read_text())
    for tie in (sm.get("ties") or sm.get("matches") or []):
        fx = tie.get("fixture") or ""
        ko = tie.get("kickoff_utc") or tie.get("kickoff")
        if " vs " not in fx or not ko:
            continue
        home, away = fx.split(" vs ", 1)
        kot = mtch._parse_ts(ko)
        if not kot:
            continue
        sb_rows.append({
            "source": "schedule_ref", "source_market_id": f"sched:{fx}:{ko}",
            "home": home, "away": away, "kickoff_utc": kot,
            "market_type": "1x2", "line": None, "period": "FT",
            "settlement": ids.S_90MIN})
sb_side = pl.DataFrame(sb_rows) if sb_rows else pl.DataFrame()
print(f"side B: {sb_side.height} markets — {sb_source}")
sb_side.tail(5).to_pandas() if sb_side.height else None

side B: 24 markets — theoddsapi silver (notebook 01)


,source,source_market_id,home,away,kickoff_utc,market_type,line,period,settlement
0,theoddsapi,wc2026:argentina__egypt__2026-07-07T16Z|h2h_la...,argentina,egypt,2026-07-07 16:00:00+00:00,h2h_lay,NaN,FT,90min
1,theoddsapi,wc2026:argentina__egypt__2026-07-07T16Z|1x2||F...,argentina,egypt,2026-07-07 16:00:00+00:00,1x2,NaN,FT,90min
2,theoddsapi,wc2026:brazil__norway__2026-07-05T20Z|totals|2...,brazil,norway,2026-07-05 20:00:00+00:00,totals,2.5,FT,90min
3,theoddsapi,wc2026:mexico__england__2026-07-06T00Z|h2h_lay...,mexico,england,2026-07-06 00:00:00+00:00,h2h_lay,NaN,FT,90min
4,theoddsapi,wc2026:switzerland__colombia__2026-07-07T20Z|t...,switzerland,colombia,2026-07-07 20:00:00+00:00,totals,2.5,FT,90min


## 3 · Manual override file
`overrides.yaml` pins or bans pairs by exact source IDs; it always wins
and the verdict reason records it. Created with a template if absent.

In [4]:
if not mtch.OVERRIDES_PATH.exists():
    mtch.OVERRIDES_PATH.write_text(
        "# Manual match overrides — exact source_market_ids, verdict accept|reject\n"
        "# pairs:\n"
        "#   - a: fifwc-esp-aut-2026-07-02\n"
        "#     b: wc2026:spain__austria__2026-07-02T19Z|1x2||FT|90min\n"
        "#     verdict: accept\n"
        "pairs: []\n")
print(mtch.OVERRIDES_PATH.read_text())

# Manual match overrides — exact source_market_ids, verdict accept|reject
# pairs:
#   - a: fifwc-esp-aut-2026-07-02
#     b: wc2026:spain__austria__2026-07-02T19Z|1x2||FT|90min
#     verdict: accept
pairs: []



## 4 · Run the matcher — every considered pair gets a verdict + reasons

In [5]:
links = (mtch.match_frames(pm_side, sb_side,
                           kickoff_tolerance_h=p.kickoff_tolerance_h,
                           min_confidence=p.min_match_confidence)
         if pm_side.height and sb_side.height else pl.DataFrame())
if links.height:
    counts = links.group_by("verdict").agg(pl.len().alias("n")).sort("verdict")
    print(counts.to_pandas().to_string(index=False))
else:
    print("no candidate pairs this run (side B empty?)")

 verdict  n
accepted  6
rejected 12


In [6]:
if links.height:
    display(links.filter(pl.col("verdict") == "accepted").head(10).to_pandas())
    display(links.filter(pl.col("verdict") == "rejected").head(10).to_pandas())
    amb = links.filter(pl.col("verdict") == "ambiguous")
    print(f"ambiguous: {amb.height}")
    if amb.height:
        display(amb.head(10).to_pandas())

,a_source,a_market_id,b_source,b_market_id,event_a,market_type_a,market_type_b,verdict,confidence,reasons
0,polymarket,fifwc-bra-nor-2026-07-05,theoddsapi,wc2026:brazil__norway__2026-07-05T20Z|1x2||FT|...,Brazil vs Norway,1x2,1x2,accepted,1.0,all gates passed
1,polymarket,fifwc-can-mar-2026-07-04,theoddsapi,wc2026:canada__morocco__2026-07-04T17Z|1x2||FT...,Canada vs Morocco,1x2,1x2,accepted,1.0,all gates passed
2,polymarket,fifwc-mex-eng-2026-07-05,theoddsapi,wc2026:mexico__england__2026-07-06T00Z|1x2||FT...,Mexico vs England,1x2,1x2,accepted,1.0,all gates passed
3,polymarket,fifwc-par-fra-2026-07-04,theoddsapi,wc2026:paraguay__france__2026-07-04T21Z|1x2||F...,Paraguay vs France,1x2,1x2,accepted,1.0,all gates passed
4,polymarket,fifwc-prt-esp-2026-07-06,theoddsapi,wc2026:portugal__spain__2026-07-06T19Z|1x2||FT...,Portugal vs Spain,1x2,1x2,accepted,1.0,all gates passed
5,polymarket,fifwc-usa-bel-2026-07-06,theoddsapi,wc2026:united-states__belgium__2026-07-07T00Z|...,United States vs Belgium,1x2,1x2,accepted,1.0,all gates passed


,a_source,a_market_id,b_source,b_market_id,event_a,market_type_a,market_type_b,verdict,confidence,reasons
0,polymarket,fifwc-bra-nor-2026-07-05,theoddsapi,wc2026:brazil__norway__2026-07-05T20Z|h2h_lay|...,Brazil vs Norway,1x2,h2h_lay,rejected,0.833,market type 1x2 vs h2h_lay
1,polymarket,fifwc-bra-nor-2026-07-05,theoddsapi,wc2026:brazil__norway__2026-07-05T20Z|totals|2...,Brazil vs Norway,1x2,totals,rejected,0.667,market type 1x2 vs totals; line None vs 2.5
2,polymarket,fifwc-can-mar-2026-07-04,theoddsapi,wc2026:canada__morocco__2026-07-04T17Z|totals|...,Canada vs Morocco,1x2,totals,rejected,0.667,market type 1x2 vs totals; line None vs 2.5
3,polymarket,fifwc-can-mar-2026-07-04,theoddsapi,wc2026:canada__morocco__2026-07-04T17Z|h2h_lay...,Canada vs Morocco,1x2,h2h_lay,rejected,0.833,market type 1x2 vs h2h_lay
4,polymarket,fifwc-mex-eng-2026-07-05,theoddsapi,wc2026:mexico__england__2026-07-06T00Z|totals|...,Mexico vs England,1x2,totals,rejected,0.667,market type 1x2 vs totals; line None vs 2.5
5,polymarket,fifwc-mex-eng-2026-07-05,theoddsapi,wc2026:mexico__england__2026-07-06T00Z|h2h_lay...,Mexico vs England,1x2,h2h_lay,rejected,0.833,market type 1x2 vs h2h_lay
6,polymarket,fifwc-par-fra-2026-07-04,theoddsapi,wc2026:paraguay__france__2026-07-04T21Z|h2h_la...,Paraguay vs France,1x2,h2h_lay,rejected,0.833,market type 1x2 vs h2h_lay
7,polymarket,fifwc-par-fra-2026-07-04,theoddsapi,wc2026:paraguay__france__2026-07-04T21Z|totals...,Paraguay vs France,1x2,totals,rejected,0.667,market type 1x2 vs totals; line None vs 2.5
8,polymarket,fifwc-prt-esp-2026-07-06,theoddsapi,wc2026:portugal__spain__2026-07-06T19Z|h2h_lay...,Portugal vs Spain,1x2,h2h_lay,rejected,0.833,market type 1x2 vs h2h_lay
9,polymarket,fifwc-prt-esp-2026-07-06,theoddsapi,wc2026:portugal__spain__2026-07-06T19Z|totals|...,Portugal vs Spain,1x2,totals,rejected,0.667,market type 1x2 vs totals; line None vs 2.5


ambiguous: 0


## 5 · The settlement-basis firewall, demonstrated on real rows
A PM advancement market for the same two teams on the same day must be
REJECTED against a 90-minute 1X2 — even though team names + kickoff agree.

In [7]:
adv_row = None
for r in mk.to_dicts():
    if mtch.classify_pm_market(r) == "advance" and r.get("question"):
        adv_row = r
        break
if adv_row and sb_side.height:
    team_q = adv_row["question"]
    demo_a = {"source": "polymarket", "source_market_id": adv_row["condition_id"],
              "home": "Spain", "away": "Austria",
              "kickoff_utc": dt.datetime(2026, 7, 2, 19, 0, tzinfo=dt.timezone.utc),
              "market_type": "1x2", "line": None, "period": "FT",
              "settlement": ids.S_ETPENS}   # advancement settles ET+pens
    demo_b = {**demo_a, "source": "theoddsapi",
              "source_market_id": "book-1x2", "settlement": ids.S_90MIN}
    verdict = mtch.score_match(demo_a, demo_b)
    print(f"real advancement market: {team_q!r}")
    print(f"verdict vs a same-day 90-min 1X2: {verdict['verdict'].upper()}")
    print("reasons:", verdict["reasons"])
    assert verdict["verdict"] == "rejected"

real advancement market: 'Will Mexico reach the Round of 16 at the 2026 FIFA World Cup?'
verdict vs a same-day 90-min 1X2: REJECTED
reasons: ['settlement et-pens vs 90min (90min vs et-pens are NEVER the same market)']


## 6 · Persist gold `market_links`

In [8]:
import lib.plotting as plot
if links.height:
    st.save_dataset(links, "gold", "market_links",
                    inputs=["silver/pm_match_events", "silver/sportsbook_quotes"],
                    notebook="03", note=f"side B = {sb_source}")
    plot.save_table(links.to_pandas(), "03_market_links")
    acc = links.filter(pl.col("verdict") == "accepted")
    print(f"gold market_links saved — {acc.height} accepted links")

gold market_links saved — 6 accepted links


## 7 · Findings, caveats, next steps

* Matching is deterministic and fully auditable: every pair carries its
  per-gate checks and human-readable reasons; overrides are explicit.
* The settlement firewall (90-min vs ET+pens) is enforced structurally —
  it is part of the market ID and a hard gate, and is unit-tested.
* **Caveat:** when notebook 01 hasn't pulled live odds, side B is a
  schedule reference — links then prove identity resolution, not price
  comparability.
* **Next:** notebook 04 uses the linked (or PM-only) markets for
  convergence; notebook 08 uses accepted links for cross-venue checks.